# 🧶 FiberNet Tutorial: Metamaterial Design Pipeline

**From Structure Generation → Mechanical Simulation → Machine Learning**

This tutorial walks you through the complete FiberNet workflow:

1. **Installation** — Set up FiberNet on your machine
2. **Structure Generation** — Create various metamaterial architectures
3. **Visualization** — Beautiful 2D/3D plots for presentations
4. **Parametric Study** — Sweep geometric parameters
5. **FEM Simulation** — Compute effective mechanical properties
6. **Machine Learning** — Train a surrogate model for property prediction

> **Note**: This tutorial uses CPU-only computation. All visualizations use `matplotlib` for maximum portability.

> **Language**: Markdown cells are in English with Chinese annotations where helpful.


## 1. Installation / 安装

### Option A: Install from GitHub (Recommended)

```bash
# Basic installation (core functionality)
pip install git+https://github.com/GellmanSparrowS/fibernet.git

# Full installation (visualization + ML)
pip install "git+https://github.com/GellmanSparrowS/fibernet.git#egg=fibernet[full]"
```

### Option B: Install from local clone

```bash
git clone https://github.com/GellmanSparrowS/fibernet.git
cd fibernet
pip install -e ".[dev,full]"
```

### Option C: If you already have the repo cloned

```bash
cd /path/to/fibernet
pip install -e .
```

### Verify Installation

Run the cell below to check everything is working:


In [ ]:
# ============================================================
# Verify FiberNet installation
# ============================================================
import sys
print(f"Python: {sys.version}")

try:
    import fibernet
    print(f"✅ FiberNet installed successfully")
except ImportError:
    print("❌ FiberNet not found. Run: pip install -e .")

try:
    import numpy as np
    import scipy
    import matplotlib
    print(f"✅ NumPy {np.__version__}, SciPy {scipy.__version__}, Matplotlib {matplotlib.__version__}")
except ImportError as e:
    print(f"❌ Missing dependency: {e}")

try:
    import sklearn
    print(f"✅ scikit-learn {sklearn.__version__}")
except ImportError:
    print("⚠️  scikit-learn not installed (needed for ML section). Run: pip install scikit-learn")

# Set matplotlib backend
import matplotlib
matplotlib.use('Agg')  # Use 'Agg' for headless/server environments
import matplotlib.pyplot as plt
print(f"✅ Matplotlib backend: {matplotlib.get_backend()}")


## 2. Core Concepts / 核心概念

FiberNet models fiber networks as collections of:

| Object | Description |
|--------|-------------|
| `Fiber` | A single fiber defined by its centerline, radius, and material |
| `FiberNetwork` | A collection of fibers with crosslinks and boundary conditions |
| `Material` | Material properties (Young's modulus, Poisson's ratio, density) |
| `Crosslink` | A connection point between two fibers |

```
FiberNetwork
├── fibers: List[Fiber]          # Each has centerline (Nx3), radius, material
├── crosslinks: List[Crosslink]  # Bonded/welded connections
├── box_size: np.ndarray         # Bounding box dimensions
├── dimension: int               # 2 or 3
└── metadata: dict               # Generator info, parameters
```


In [ ]:
from fibernet.core.fiber import Fiber
from fibernet.core.network import FiberNetwork
from fibernet.core.material import Material

# Create a simple steel fiber
steel = Material(name="steel", youngs_modulus=200e9, density=7800, poissons_ratio=0.3)
fiber = Fiber.straight([0, 0, 0], [10, 0, 0], radius=0.1, material=steel)

print(f"Fiber length: {fiber.length:.2f}")
print(f"Cross-section area: {fiber.cross_section_area:.4f}")
print(f"Material: {fiber.material.name}, E = {fiber.material.youngs_modulus:.0e} Pa")


## 3. Visualization Setup / 可视化设置

We'll create reusable plotting functions for clean, publication-quality figures.
All plots use `matplotlib` only (no PyVista required).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from mpl_toolkits.mplot3d.art3d import Line3DCollection
import os

# Create output directory for figures and data
os.makedirs("fibernet_output", exist_ok=True)
os.makedirs("fibernet_output/structures", exist_ok=True)
os.makedirs("fibernet_output/results", exist_ok=True)
os.makedirs("fibernet_output/dataset", exist_ok=True)

# ============================================================
# Plotting helpers
# ============================================================

def plot_network_2d(net, title="", save_path=None, figsize=(8, 8), 
                    color_by="material", cmap="Set2", show_nodes=True,
                    ax=None, **kwargs):
    """Plot a 2D fiber network with publication-quality rendering."""
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=figsize)
    
    colors = plt.cm.get_cmap(cmap)
    
    for i, fiber in enumerate(net.fibers):
        pts = fiber.centerline
        x, y = pts[:, 0], pts[:, 1]
        
        # Color by fiber index
        c = colors(i % 8)
        
        # Line width proportional to radius
        lw = max(0.5, fiber.radius * 20)
        ax.plot(x, y, color=c, linewidth=lw, solid_capstyle='round', alpha=0.85)
    
    # Plot crosslink nodes
    if show_nodes and net.crosslinks:
        cx = [cl.position[0] for cl in net.crosslinks]
        cy = [cl.position[1] for cl in net.crosslinks]
        ax.scatter(cx, cy, c='black', s=8, zorder=5, alpha=0.6)
    
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=10)
    ax.set_xlabel("x", fontsize=12)
    ax.set_ylabel("y", fontsize=12)
    ax.grid(True, alpha=0.15)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight', 
                   facecolor='white', edgecolor='none')
        print(f"  💾 Saved: {save_path}")
    return ax


def plot_network_3d(net, title="", save_path=None, figsize=(10, 10),
                    cmap="Set2", elevation=25, azimuth=135):
    """Plot a 3D fiber network."""
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='3d')
    
    colors = plt.cm.get_cmap(cmap)
    
    for i, fiber in enumerate(net.fibers):
        pts = fiber.centerline
        c = colors(i % 8)
        lw = max(0.5, fiber.radius * 15)
        ax.plot(pts[:, 0], pts[:, 1], pts[:, 2], 
                color=c, linewidth=lw, alpha=0.8)
    
    ax.set_title(title, fontsize=14, fontweight='bold', pad=10)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.view_init(elev=elevation, azim=azimuth)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight',
                   facecolor='white', edgecolor='none')
        print(f"  💾 Saved: {save_path}")
    return ax


def plot_deformed_2d(net, result, title="", save_path=None, 
                     scale=50, figsize=(10, 8)):
    """Plot undeformed vs deformed network."""
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # Undeformed
    for fiber in net.fibers:
        pts = fiber.centerline
        axes[0].plot(pts[:, 0], pts[:, 1], 'b-', linewidth=1, alpha=0.5)
    axes[0].set_title("Undeformed", fontsize=13)
    axes[0].set_aspect('equal')
    axes[0].grid(True, alpha=0.15)
    
    # Deformed
    from fibernet.sim.mechanical import FiberFEM
    fem = FiberFEM(net, segments_per_fiber=5)
    
    for fiber in net.fibers:
        pts = fiber.centerline
        deformed_pts = pts.copy()
        
        if result.displacements is not None:
            for k, pt in enumerate(pts):
                for n_idx in range(fem.num_nodes):
                    if np.linalg.norm(fem.node_positions[n_idx] - pt) < 1e-6:
                        for d in range(min(2, net.dimension)):
                            dof = n_idx * 6 + d
                            if dof < len(result.displacements):
                                deformed_pts[k, d] += scale * result.displacements[dof]
                        break
        
        axes[1].plot(deformed_pts[:, 0], deformed_pts[:, 1], 'r-', 
                     linewidth=1, alpha=0.7)
    
    axes[1].set_title(f"Deformed (scale={scale}×)", fontsize=13)
    axes[1].set_aspect('equal')
    axes[1].grid(True, alpha=0.15)
    
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight',
                   facecolor='white', edgecolor='none')
        print(f"  💾 Saved: {save_path}")
    return axes

print("✅ Visualization helpers defined")


## 4. Metamaterial Structure Gallery / 超材料结构展示

Let's generate and visualize a variety of metamaterial architectures.
Each structure is saved as both **PNG** (for presentations) and **JSON** (for later simulation).

> **Tip**: The `metadata` dict on each network records the generator and parameters used,
> making results fully reproducible.


### 4.1 Re-entrant Honeycomb (Auxetic) / 反入蜂窝

The re-entrant honeycomb is the classic auxetic structure. By making cell walls point **inward**
(re-entrant angle > 120°), the structure expands laterally when stretched — a **negative
Poisson's ratio**.

Key parameters:
- `reentrant_angle`: Controls auxetic behavior (150° = strong auxetic, 120° = hexagonal)
- `cell_height / cell_width`: Unit cell dimensions
- `grid_size`: Number of cells in each direction


In [ ]:
from fibernet.gen.metamaterials import reentrant_honeycomb_2d

# Generate re-entrant auxetic honeycomb
net_reentrant = reentrant_honeycomb_2d(
    reentrant_angle=150,      # Strongly auxetic
    cell_height=10, cell_width=10,
    grid_size=(6, 6),
    radius=0.2,
)

print(f"Re-entrant honeycomb: {net_reentrant.num_fibers} fibers, "
      f"{net_reentrant.num_crosslinks} crosslinks")
print(f"Metadata: {net_reentrant.metadata}")

# Plot and save
plot_network_2d(net_reentrant, 
                title="Re-entrant Auxetic Honeycomb (θ=150°)",
                save_path="fibernet_output/structures/reentrant_honeycomb.png")

# Save structure to JSON for later use
net_reentrant.save_json("fibernet_output/structures/reentrant_honeycomb.json")
print("  💾 Saved JSON: fibernet_output/structures/reentrant_honeycomb.json")
plt.show()


### 4.2 Angle Sweep: Hexagonal → Auxetic

See how the re-entrant angle changes the structure geometry:


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

angles = [100, 120, 150]
titles = ["Concave (θ=100°)", "Hexagonal (θ=120°)", "Auxetic (θ=150°)"]

for ax, angle, title in zip(axes, angles, titles):
    net = reentrant_honeycomb_2d(
        reentrant_angle=angle,
        cell_height=10, cell_width=10,
        grid_size=(4, 4), radius=0.2,
    )
    plot_network_2d(net, title=title, ax=ax, show_nodes=False)

plt.tight_layout()
plt.savefig("fibernet_output/structures/angle_comparison.png", 
            dpi=200, bbox_inches='tight', facecolor='white')
print("  💾 Saved: fibernet_output/structures/angle_comparison.png")
plt.show()


### 4.3 Chiral Honeycomb (Node-Ligament) / 手性蜂窝

Each unit cell has a **circular node** connected to neighbors via tangential ligaments.
This produces in-plane auxetic behavior through a rotation mechanism.

Reference: Spadoni & Ruzzene, *J. Intell. Mat. Syst. Str.* 2012


In [ ]:
from fibernet.gen.metamaterials import chiral_honeycomb_2d

net_chiral = chiral_honeycomb_2d(
    node_radius=3.0,
    ligament_length=6.0,
    grid_size=(4, 4),
    fiber_radius=0.15,
    num_node_points=24,
)

print(f"Chiral honeycomb: {net_chiral.num_fibers} fibers")

plot_network_2d(net_chiral,
                title="Chiral Honeycomb (Node-Ligament Structure)",
                save_path="fibernet_output/structures/chiral_honeycomb.png")

net_chiral.save_json("fibernet_output/structures/chiral_honeycomb.json")
plt.show()


### 4.4 Star-Shaped Auxetic / 星形拉胀

Star-shaped unit cells with inward-pointing arms. Tunable via arm count and inner angle.


In [ ]:
from fibernet.gen.metamaterials import star_honeycomb_2d

net_star = star_honeycomb_2d(
    star_arm_length=5.0,
    star_inner_angle=60,
    grid_size=(4, 4),
    num_arms=6,
    radius=0.15,
)

print(f"Star honeycomb: {net_star.num_fibers} fibers")

plot_network_2d(net_star,
                title="Star-Shaped Auxetic Honeycomb (6-arm)",
                save_path="fibernet_output/structures/star_honeycomb.png")

net_star.save_json("fibernet_output/structures/star_honeycomb.json")
plt.show()


### 4.5 Arrowhead Auxetic / 箭头拉胀

The arrowhead (fishbone) geometry produces strong auxetic behavior with
a large negative Poisson's ratio range.


In [ ]:
from fibernet.gen.metamaterials import arrowhead_auxetic_2d

net_arrow = arrowhead_auxetic_2d(
    arm_length=8.0,
    arm_angle=60,
    grid_size=(5, 5),
    radius=0.15,
)

print(f"Arrowhead auxetic: {net_arrow.num_fibers} fibers")

plot_network_2d(net_arrow,
                title="Arrowhead Auxetic Structure",
                save_path="fibernet_output/structures/arrowhead_auxetic.png")

net_arrow.save_json("fibernet_output/structures/arrowhead_auxetic.json")
plt.show()


### 4.6 Hierarchical Lattice / 层次格子

Self-similar structure: each strut is replaced by a smaller copy of the parent pattern.
Creates multi-scale architectures with unusual mechanical scaling.


In [ ]:
from fibernet.gen.metamaterials import hierarchical_lattice_2d

net_hier = hierarchical_lattice_2d(
    base_type="triangular",
    levels=2,
    cell_size=50,
    radius=0.3,
    radius_decay=0.6,
)

print(f"Hierarchical lattice (2 levels): {net_hier.num_fibers} fibers")

plot_network_2d(net_hier,
                title="Hierarchical Triangular Lattice (Level 2)",
                save_path="fibernet_output/structures/hierarchical_lattice.png")

net_hier.save_json("fibernet_output/structures/hierarchical_lattice.json")
plt.show()


### 4.7 3D Structures: Octet Truss & Diamond Lattice

3D architectures are plotted with matplotlib 3D projection:

- **Octet Truss**: Stretch-dominated (high stiffness), 12 struts per cell, FCC topology
- **Diamond Lattice**: Bending-dominated (isotropic), 4-coordinated tetrahedral nodes


In [ ]:
from fibernet.gen.metamaterials import proper_octet_truss_3d, diamond_lattice_3d

# Octet truss
net_octet = proper_octet_truss_3d(
    spacing=10, grid_size=(2, 2, 2), radius=0.25,
)
print(f"Octet truss: {net_octet.num_fibers} struts")

plot_network_3d(net_octet, 
                title="Octet Truss (Stretch-Dominated)",
                save_path="fibernet_output/structures/octet_truss_3d.png")
plt.show()

# Diamond lattice
net_diamond = diamond_lattice_3d(
    spacing=10, grid_size=(2, 2, 2), radius=0.2,
)
print(f"Diamond lattice: {net_diamond.num_fibers} struts")

plot_network_3d(net_diamond,
                title="Diamond Lattice (Bending-Dominated)",
                save_path="fibernet_output/structures/diamond_lattice_3d.png")
plt.show()

# Save JSON
net_octet.save_json("fibernet_output/structures/octet_truss_3d.json")
net_diamond.save_json("fibernet_output/structures/diamond_lattice_3d.json")


### 4.8 Woven & Braided Structures / 编织结构


In [ ]:
from fibernet.gen.woven import plain_weave_2d, twill_weave_2d
from fibernet.gen.chiral import braided_rope, twisted_bundle

# Plain weave
net_weave = plain_weave_2d(spacing=2, grid_size=(8, 8), amplitude=0.3, radius=0.08)
print(f"Plain weave: {net_weave.num_fibers} fibers")

# Braided rope
net_braid = braided_rope(num_strands=5, rope_radius=3, pitch=15, num_turns=3, fiber_radius=0.2)
print(f"Braided rope: {net_braid.num_fibers} strands")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
plot_network_2d(net_weave, title="Plain Weave (1/1)", ax=axes[0], show_nodes=False)
plot_network_3d(net_braid, title="5-Strand Braided Rope", 
                save_path="fibernet_output/structures/braided_rope_3d.png")
plt.show()

net_weave.save_json("fibernet_output/structures/plain_weave.json")
net_braid.save_json("fibernet_output/structures/braided_rope.json")


### 4.9 Structure Gallery Summary / 结构总结


In [ ]:
# Generate a summary comparison figure
structures = {
    "Re-entrant (θ=150°)": lambda: reentrant_honeycomb_2d(reentrant_angle=150, grid_size=(4,4), radius=0.2),
    "Chiral Honeycomb": lambda: chiral_honeycomb_2d(grid_size=(3,3), node_radius=3, ligament_length=6, num_node_points=20),
    "Star (6-arm)": lambda: star_honeycomb_2d(grid_size=(3,3), num_arms=6, radius=0.15),
    "Arrowhead": lambda: arrowhead_auxetic_2d(grid_size=(4,4), radius=0.15),
}

fig, axes = plt.subplots(2, 2, figsize=(16, 16))
axes_flat = axes.flatten()

for (name, gen_func), ax in zip(structures.items(), axes_flat):
    net = gen_func()
    plot_network_2d(net, title=f"{name}\n({net.num_fibers} fibers)", 
                    ax=ax, show_nodes=False, cmap="Set2")

plt.suptitle("Metamaterial Structure Gallery", fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("fibernet_output/structures/gallery_summary.png",
            dpi=200, bbox_inches='tight', facecolor='white')
print("  💾 Saved: fibernet_output/structures/gallery_summary.png")
plt.show()


## 5. Parametric Study: Re-entrant Honeycomb / 参数研究

We'll focus on the **re-entrant honeycomb** because:

1. **Clear physics**: The re-entrant angle directly controls Poisson's ratio
2. **Well-studied**: Analytical models exist for validation (Gibson & Ashby)
3. **Rich behavior**: Transitions from positive to negative Poisson's ratio
4. **Student-friendly**: Easy to explain in presentations

### Study Plan

| Parameter | Range | Steps |
|-----------|-------|-------|
| Re-entrant angle θ | 100°–170° | 15 values |
| Grid size | (5,5) fixed | - |
| Cell size | 10×10 fixed | - |
| Fiber radius | 0.2 fixed | - |

For each angle, we compute:
- Effective Young's modulus $E^*$
- Poisson's ratio $\nu^*$
- Relative density $\rho^*$
- Total strain energy


In [ ]:
from fibernet.gen.metamaterials import reentrant_honeycomb_2d
from fibernet.sim.mechanical import FiberFEM, compute_effective_properties, poisson_ratio

# ============================================================
# Parametric sweep: re-entrant angle
# ============================================================

angles = np.linspace(100, 170, 15)  # 15 angles from 100° to 170°
results = []

print("Computing effective properties for each angle...")
print(f"{'Angle':>8} {'E_x':>12} {'E_y':>12} {'ν_xy':>8} {'ρ*':>8} {'Fibers':>8}")
print("-" * 60)

for angle in angles:
    net = reentrant_honeycomb_2d(
        reentrant_angle=angle,
        cell_height=10, cell_width=10,
        grid_size=(5, 5),
        radius=0.2,
    )
    
    # Compute effective properties
    props = compute_effective_properties(net, strain=0.001, segments_per_fiber=3)
    
    # Direct Poisson's ratio computation
    nu_xy = poisson_ratio(net, strain=0.001, loading_axis=0, transverse_axis=1,
                          segments_per_fiber=3)
    
    row = {
        "angle": angle,
        "E_x": props.E_x,
        "E_y": props.E_y,
        "E_z": props.E_z,
        "nu_xy": nu_xy,
        "nu_xz": props.nu_xz,
        "relative_density": props.relative_density,
        "num_fibers": net.num_fibers,
        "num_crosslinks": net.num_crosslinks,
        "energy": props.E_x * 0.001**2 / 2,  # approximate strain energy
    }
    results.append(row)
    
    print(f"{angle:>7.1f}° {props.E_x:>12.4e} {props.E_y:>12.4e} "
          f"{nu_xy:>+8.4f} {props.relative_density:>8.4f} {net.num_fibers:>8d}")

print(f"\n✅ Computed {len(results)} configurations")


### 5.1 Results Visualization / 结果可视化


In [ ]:
import pandas as pd

df = pd.DataFrame(results)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) Young's modulus vs angle
ax = axes[0, 0]
ax.plot(df['angle'], df['E_x'], 'bo-', linewidth=2, markersize=8, label='E_x')
ax.plot(df['angle'], df['E_y'], 'rs-', linewidth=2, markersize=8, label='E_y')
ax.set_xlabel("Re-entrant Angle θ (degrees)", fontsize=12)
ax.set_ylabel("Effective Modulus E* (Pa)", fontsize=12)
ax.set_title("(a) Effective Young's Modulus", fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# (b) Poisson's ratio vs angle
ax = axes[0, 1]
ax.plot(df['angle'], df['nu_xy'], 'go-', linewidth=2, markersize=8)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axhspan(min(df['nu_xy'].min(), -0.1), 0, alpha=0.1, color='red', label='Auxetic region')
ax.set_xlabel("Re-entrant Angle θ (degrees)", fontsize=12)
ax.set_ylabel("Poisson's Ratio ν_xy", fontsize=12)
ax.set_title("(b) Poisson's Ratio", fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# (c) Relative density vs angle
ax = axes[1, 0]
ax.plot(df['angle'], df['relative_density'], 'm^-', linewidth=2, markersize=8)
ax.set_xlabel("Re-entrant Angle θ (degrees)", fontsize=12)
ax.set_ylabel("Relative Density ρ*", fontsize=12)
ax.set_title("(c) Relative Density", fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

# (d) Stiffness vs Poisson's ratio (property space)
ax = axes[1, 1]
sc = ax.scatter(df['nu_xy'], df['E_x'], c=df['angle'], cmap='viridis',
                s=100, edgecolors='black', linewidth=0.5)
plt.colorbar(sc, ax=ax, label="Angle θ (°)")
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel("Poisson's Ratio ν_xy", fontsize=12)
ax.set_ylabel("Effective Modulus E_x (Pa)", fontsize=12)
ax.set_title("(d) Property Space (ν vs E*)", fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.suptitle("Re-entrant Honeycomb: Parametric Study", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("fibernet_output/results/parametric_study.png",
            dpi=200, bbox_inches='tight', facecolor='white')
print("  💾 Saved: fibernet_output/results/parametric_study.png")
plt.show()


### 5.2 Save Dataset / 保存数据集


In [ ]:
import json

# Save as JSON
dataset_path = "fibernet_output/dataset/reentrant_parametric.json"
with open(dataset_path, 'w') as f:
    json.dump({
        "description": "Re-entrant honeycomb parametric study",
        "generator": "reentrant_honeycomb_2d",
        "parameters": {
            "cell_height": 10, "cell_width": 10,
            "grid_size": [5, 5], "radius": 0.2,
        },
        "swept_variable": "reentrant_angle",
        "results": results,
    }, f, indent=2)

# Also save as CSV for easy loading
csv_path = "fibernet_output/dataset/reentrant_parametric.csv"
df.to_csv(csv_path, index=False)

print(f"  💾 Saved JSON: {dataset_path}")
print(f"  💾 Saved CSV: {csv_path}")
print(f"  📊 Dataset: {len(results)} configurations")


## 6. Detailed FEM Analysis / 有限元分析

Let's do a deeper mechanical analysis on one specific structure:

1. **Stress-strain curve** — Apply incremental strain
2. **Deformation visualization** — Show undeformed vs deformed
3. **Energy analysis** — Strain energy vs applied strain


In [ ]:
from fibernet.sim.mechanical import FiberFEM, stress_strain_curve

# Pick a strongly auxetic structure
net = reentrant_honeycomb_2d(
    reentrant_angle=155,
    cell_height=10, cell_width=10,
    grid_size=(5, 5), radius=0.2,
)

print(f"Structure: Re-entrant honeycomb (θ=155°)")
print(f"Fibers: {net.num_fibers}, Crosslinks: {net.num_crosslinks}")

# ============================================================
# Compute stress-strain curve
# ============================================================
print("\nComputing stress-strain curve...")
strains, stresses = stress_strain_curve(
    net, max_strain=0.05, num_steps=10, axis=0, segments_per_fiber=5,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stress-strain curve
ax = axes[0]
ax.plot(strains * 100, stresses, 'b-o', linewidth=2, markersize=6)
ax.set_xlabel("Strain ε (%)", fontsize=12)
ax.set_ylabel("Stress σ (Pa)", fontsize=12)
ax.set_title("Stress-Strain Curve (x-direction)", fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

# Effective modulus from initial slope
if len(strains) > 1:
    E_eff = (stresses[1] - stresses[0]) / (strains[1] - strains[0])
    ax.annotate(f'E* = {E_eff:.2e} Pa', xy=(strains[2]*100, stresses[2]),
                fontsize=11, color='red',
                arrowprops=dict(arrowstyle='->', color='red'),
                xytext=(strains[4]*100, stresses[4]*1.2))

# Strain energy vs strain
ax = axes[1]
energy = 0.5 * stresses * strains
ax.plot(strains * 100, energy, 'r-s', linewidth=2, markersize=6)
ax.set_xlabel("Strain ε (%)", fontsize=12)
ax.set_ylabel("Strain Energy Density (J/m³)", fontsize=12)
ax.set_title("Strain Energy vs Deformation", fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("fibernet_output/results/stress_strain_curve.png",
            dpi=200, bbox_inches='tight', facecolor='white')
print("  💾 Saved: fibernet_output/results/stress_strain_curve.png")
plt.show()


### 6.1 Deformation Visualization / 变形可视化


In [ ]:
# Apply a visible strain and show deformation
fem = FiberFEM(net, segments_per_fiber=5)
result = fem.apply_uniaxial_strain(strain=0.02, axis=0)

print(f"Max displacement: {result.max_displacement():.4f}")
print(f"Total strain energy: {result.energy:.6e} J")
print(f"Max stress: {result.max_stress():.4e} Pa")

plot_deformed_2d(net, result, 
                  title="Re-entrant Honeycomb Under 2% Uniaxial Strain",
                  scale=10,
                  save_path="fibernet_output/results/deformation_visualization.png")
plt.show()


## 7. Machine Learning: Surrogate Model / 机器学习代理模型

Now we'll train a machine learning model to **predict mechanical properties**
from structural parameters, avoiding expensive FEM computations.

### Pipeline

```
Structure Parameters → Feature Extraction → ML Model → Property Prediction
(θ, h, l, r, grid)    (geometric stats)    (RF/NN)     (E*, ν*, ρ*)
```

### Step 1: Expand the dataset

We need more data for ML. Let's generate a larger parametric dataset
by varying multiple parameters simultaneously.


In [ ]:
# ============================================================
# Generate expanded dataset for ML
# ============================================================
from itertools import product

# Parameter space
param_angles = np.linspace(100, 170, 10)     # 10 angles
param_heights = [8, 10, 12]                   # 3 heights
param_widths = [8, 10, 12]                    # 3 widths
param_radii = [0.15, 0.2, 0.25, 0.3]        # 4 radii

print(f"Total configurations: {len(param_angles) * len(param_heights) * len(param_widths) * len(param_radii)}")

ml_dataset = []
count = 0

for angle, h, w, r in product(param_angles, param_heights, param_widths, param_radii):
    try:
        net = reentrant_honeycomb_2d(
            reentrant_angle=angle,
            cell_height=h, cell_width=w,
            grid_size=(4, 4), radius=r,
        )
        
        props = compute_effective_properties(net, strain=0.001, segments_per_fiber=3)
        nu = poisson_ratio(net, strain=0.001, loading_axis=0, transverse_axis=1,
                           segments_per_fiber=3)
        
        ml_dataset.append({
            "angle": angle,
            "cell_height": h,
            "cell_width": w,
            "radius": r,
            "num_fibers": net.num_fibers,
            "num_crosslinks": net.num_crosslinks,
            "relative_density": props.relative_density,
            "E_x": props.E_x,
            "E_y": props.E_y,
            "nu_xy": nu,
        })
        count += 1
        
        if count % 30 == 0:
            print(f"  Progress: {count} configurations computed...")
            
    except Exception as e:
        print(f"  ⚠️  Skipped (θ={angle:.0f}, h={h}, w={w}, r={r}): {e}")

print(f"\n✅ Dataset: {len(ml_dataset)} configurations")

ml_df = pd.DataFrame(ml_dataset)
ml_df.to_csv("fibernet_output/dataset/ml_dataset.csv", index=False)
print(f"  💾 Saved: fibernet_output/dataset/ml_dataset.csv")


### Step 2: Feature Engineering & Model Training

We extract features from the structural parameters and train regression models.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

# ============================================================
# Prepare features and targets
# ============================================================

# Features: structural parameters + derived geometric features
feature_cols = ['angle', 'cell_height', 'cell_width', 'radius',
                'num_fibers', 'num_crosslinks', 'relative_density']
target_cols = ['E_x', 'E_y', 'nu_xy']

# Add engineered features
ml_df['angle_rad'] = np.radians(ml_df['angle'])
ml_df['h_over_w'] = ml_df['cell_height'] / ml_df['cell_width']
ml_df['cos_angle'] = np.cos(ml_df['angle_rad'])
ml_df['sin_angle'] = np.sin(ml_df['angle_rad'])

all_features = feature_cols + ['angle_rad', 'h_over_w', 'cos_angle', 'sin_angle']

X = ml_df[all_features].values
y_E = np.log10(ml_df['E_x'].values + 1)  # Log transform for better scaling
y_nu = ml_df['nu_xy'].values

# Train-test split
X_train, X_test, yE_train, yE_test, ynu_train, ynu_test = train_test_split(
    X, y_E, y_nu, test_size=0.2, random_state=42,
)

# Scale features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"Features: {len(all_features)}")


### Step 3: Train Multiple Models

We compare three approaches:
- **Random Forest**: Good baseline, interpretable feature importance
- **Gradient Boosting**: Better accuracy, still interpretable
- **Neural Network (MLP)**: Most flexible, handles nonlinearities well


In [ ]:
# ============================================================
# Model Training
# ============================================================

models = {
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42, max_depth=5),
    "Neural Network (MLP)": MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42),
}

results_ml = {}

print("=" * 70)
print(f"{'Model':<25} {'Target':<8} {'MAE':>10} {'R²':>10} {'RMSE':>10}")
print("=" * 70)

for name, model in models.items():
    for target_name, y_train, y_test in [
        ("log(E_x)", yE_train, yE_test),
        ("ν_xy", ynu_train, ynu_test),
    ]:
        model_clone = type(model)(**model.get_params())
        model_clone.fit(X_train_s, y_train)
        y_pred = model_clone.predict(X_test_s)
        
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        
        results_ml[(name, target_name)] = {
            "model": model_clone, "mae": mae, "r2": r2, "rmse": rmse,
            "y_pred": y_pred, "y_test": y_test,
        }
        
        print(f"{name:<25} {target_name:<8} {mae:>10.4f} {r2:>10.4f} {rmse:>10.4f}")

print("=" * 70)


### Step 4: Visualize ML Results


In [ ]:
# ============================================================
# ML Results Visualization
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# (a) Best model predictions vs actual - E_x
best_E_model = max(
    [(k, v) for k, v in results_ml.items() if k[1] == "log(E_x)"],
    key=lambda x: x[1]['r2']
)
ax = axes[0, 0]
y_test_e = best_E_model[1]['y_test']
y_pred_e = best_E_model[1]['y_pred']
ax.scatter(10**y_test_e, 10**y_pred_e, alpha=0.5, s=30, c='steelblue', edgecolors='black', linewidth=0.3)
lims = [min(10**y_test_e.min(), 10**y_pred_e.min()), max(10**y_test_e.max(), 10**y_pred_e.max())]
ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel("Actual E_x (Pa)", fontsize=12)
ax.set_ylabel("Predicted E_x (Pa)", fontsize=12)
ax.set_title(f"(a) E_x Prediction ({best_E_model[0][0]})\nR²={best_E_model[1]['r2']:.4f}", 
             fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# (b) Best model predictions vs actual - Poisson's ratio
best_nu_model = max(
    [(k, v) for k, v in results_ml.items() if k[1] == "ν_xy"],
    key=lambda x: x[1]['r2']
)
ax = axes[0, 1]
y_test_nu = best_nu_model[1]['y_test']
y_pred_nu = best_nu_model[1]['y_pred']
ax.scatter(y_test_nu, y_pred_nu, alpha=0.5, s=30, c='seagreen', edgecolors='black', linewidth=0.3)
lims = [min(y_test_nu.min(), y_pred_nu.min()), max(y_test_nu.max(), y_pred_nu.max())]
ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel("Actual ν_xy", fontsize=12)
ax.set_ylabel("Predicted ν_xy", fontsize=12)
ax.set_title(f"(b) ν_xy Prediction ({best_nu_model[0][0]})\nR²={best_nu_model[1]['r2']:.4f}", 
             fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# (c) Feature importance (from Random Forest)
ax = axes[1, 0]
rf_key = ("Random Forest", "log(E_x)")
rf_model = results_ml[rf_key]['model']
importances = rf_model.feature_importances_
sorted_idx = np.argsort(importances)
ax.barh(range(len(all_features)), importances[sorted_idx], color='steelblue', edgecolor='black')
ax.set_yticks(range(len(all_features)))
ax.set_yticklabels([all_features[i] for i in sorted_idx])
ax.set_xlabel("Feature Importance", fontsize=12)
ax.set_title("(c) Feature Importance (Random Forest → E_x)", fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# (d) Model comparison bar chart
ax = axes[1, 1]
model_names = list(models.keys())
r2_E = [results_ml[(m, "log(E_x)")]['r2'] for m in model_names]
r2_nu = [results_ml[(m, "ν_xy")]['r2'] for m in model_names]
x = np.arange(len(model_names))
width = 0.35
bars1 = ax.bar(x - width/2, r2_E, width, label='R² (E_x)', color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, r2_nu, width, label='R² (ν_xy)', color='seagreen', edgecolor='black')
ax.set_ylabel("R² Score", fontsize=12)
ax.set_title("(d) Model Comparison", fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=10)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle("ML Surrogate Model Results", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("fibernet_output/results/ml_results.png",
            dpi=200, bbox_inches='tight', facecolor='white')
print("  💾 Saved: fibernet_output/results/ml_results.png")
plt.show()


### Step 5: Use the ML Model for Rapid Screening

With a trained model, we can rapidly screen thousands of designs:


In [ ]:
# ============================================================
# Use trained model for rapid design screening
# ============================================================

# Generate candidate designs
n_candidates = 500
candidate_params = {
    "angle": np.random.uniform(100, 170, n_candidates),
    "cell_height": np.random.choice([8, 10, 12, 14], n_candidates),
    "cell_width": np.random.choice([8, 10, 12, 14], n_candidates),
    "radius": np.random.choice([0.15, 0.2, 0.25, 0.3], n_candidates),
}

# Estimate geometric features (approximate)
candidate_params["angle_rad"] = np.radians(candidate_params["angle"])
candidate_params["h_over_w"] = candidate_params["cell_height"] / candidate_params["cell_width"]
candidate_params["cos_angle"] = np.cos(candidate_params["angle_rad"])
candidate_params["sin_angle"] = np.sin(candidate_params["angle_rad"])
# Approximate num_fibers and crosslinks (based on grid size (4,4))
candidate_params["num_fibers"] = 100  # approximate
candidate_params["num_crosslinks"] = 50  # approximate
candidate_params["relative_density"] = 0.1  # approximate

X_candidates = np.column_stack([
    candidate_params[f] for f in all_features
])
X_candidates_s = scaler.transform(X_candidates)

# Predict with best models
best_rf_E = results_ml[("Random Forest", "log(E_x)")]['model']
best_rf_nu = results_ml[("Random Forest", "ν_xy")]['model']

pred_E = 10 ** best_rf_E.predict(X_candidates_s)
pred_nu = best_rf_nu.predict(X_candidates_s)

# ============================================================
# Pareto front: maximize stiffness AND auxetic behavior
# ============================================================

fig, ax = plt.subplots(figsize=(10, 8))

sc = ax.scatter(pred_nu, pred_E, c=candidate_params["angle"], cmap='viridis',
                s=40, alpha=0.6, edgecolors='black', linewidth=0.3)
plt.colorbar(sc, ax=ax, label="Re-entrant Angle θ (°)")

ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7, label='ν=0 boundary')

# Highlight auxetic + high stiffness candidates
auxetic_mask = pred_nu < -0.1
high_stiff_mask = pred_E > np.percentile(pred_E, 75)
best_mask = auxetic_mask & high_stiff_mask

if np.any(best_mask):
    ax.scatter(pred_nu[best_mask], pred_E[best_mask], 
               c='red', s=120, marker='*', edgecolors='black', linewidth=1,
               label=f'Best candidates ({np.sum(best_mask)})', zorder=5)

ax.set_xlabel("Predicted Poisson's Ratio ν_xy", fontsize=13)
ax.set_ylabel("Predicted E_x (Pa)", fontsize=13)
ax.set_title("Rapid Design Screening: Stiffness vs Auxetic Behavior\n(ML-predicted, 500 candidates)",
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("fibernet_output/results/design_screening.png",
            dpi=200, bbox_inches='tight', facecolor='white')
print("  💾 Saved: fibernet_output/results/design_screening.png")

if np.any(best_mask):
    print(f"\n🏆 Found {np.sum(best_mask)} top candidates (auxetic + stiff):")
    best_indices = np.where(best_mask)[0]
    for idx in best_indices[:5]:
        print(f"  θ={candidate_params['angle'][idx]:.1f}°, "
              f"h={candidate_params['cell_height'][idx]:.0f}, "
              f"w={candidate_params['cell_width'][idx]:.0f}, "
              f"r={candidate_params['radius'][idx]:.2f} → "
              f"ν={pred_nu[idx]:.3f}, E={pred_E[idx]:.2e}")

plt.show()


## 8. Reinforcement Learning: Closed-Loop Design Optimization / 强化学习闭环优化

Now we close the loop: use the **ML surrogate as a fast reward function**,
and a **reinforcement learning agent** to autonomously discover optimal metamaterial designs.

### Closed-Loop Pipeline

```
Generate Structure → FEM Simulate → Extract Features → Train ML Model
      ↑                                                          ↓
      └── RL Agent proposes new parameters ← Evaluate with ML ←──┘
```

**State**: Structural parameters `[θ, h, w, r]` (re-entrant angle, cell height/width, strut radius)  
**Action**: Continuous adjustments `Δθ, Δh, Δw, Δr`  
**Reward**: Multi-objective — maximize stiffness AND maximize auxetic behavior (negative Poisson's ratio)

This approach avoids expensive FEM at each step — the ML model runs in microseconds.

In [ ]:
# ============================================================
# Reinforcement Learning Environment
# Uses the trained ML surrogate as the reward function
# ============================================================

class MetamaterialRLAgent:
    """
    RL agent for metamaterial inverse design.
    
    The agent learns to propose re-entrant honeycomb parameters
    that produce target mechanical properties (high stiffness + auxetic).
    """
    
    def __init__(self, ml_model_E, ml_model_nu, scaler, feature_names):
        self.ml_E = ml_model_E   # predicts log10(E_x)
        self.ml_nu = ml_model_nu # predicts nu_xy
        self.scaler = scaler
        self.feature_names = feature_names
        
        # Parameter bounds
        self.bounds = {
            'angle': (100, 170),
            'cell_height': (6, 16),
            'cell_width': (6, 16),
            'radius': (0.1, 0.35),
        }
        self.param_keys = ['angle', 'cell_height', 'cell_width', 'radius']
        
        # Q-table for discretized actions (simplified RL)
        self.action_space = self._build_action_space(n_actions_per_dim=5)
        
    def _build_action_space(self, n_actions_per_dim=5):
        """Discretize continuous actions into a finite set."""
        from itertools import product
        steps = np.linspace(-1, 1, n_actions_per_dim)
        return list(product(*[steps]*4))
    
    def _params_to_features(self, params):
        """Convert raw params to ML feature vector."""
        feat = dict(params)
        feat['angle_rad'] = np.radians(feat['angle'])
        feat['h_over_w'] = feat['cell_height'] / feat['cell_width']
        feat['cos_angle'] = np.cos(feat['angle_rad'])
        feat['sin_angle'] = np.sin(feat['angle_rad'])
        feat['num_fibers'] = 100
        feat['num_crosslinks'] = 50
        feat['relative_density'] = 0.1
        return np.array([[feat[f] for f in self.feature_names]])
    
    def _predict_properties(self, params):
        """Use ML surrogate to predict properties."""
        X = self._params_to_features(params)
        X_s = self.scaler.transform(X)
        log_E = self.ml_E.predict(X_s)[0]
        nu = self.ml_nu.predict(X_s)[0]
        return 10 ** log_E, nu  # E in Pa, nu dimensionless
    
    def _compute_reward(self, params, target_E=None, target_nu=None):
        """
        Multi-objective reward.
        
        Default: maximize stiffness AND maximize auxetic behavior.
        Custom targets can be provided.
        """
        E, nu = self._predict_properties(params)
        
        # Stiffness reward: higher E is better
        reward_E = np.log10(E + 1e-10)
        
        # Auxetic reward: more negative nu is better
        reward_nu = -nu  # positive when auxetic
        
        # Custom target mode
        if target_E is not None:
            reward_E = -abs(E - target_E) / target_E
        if target_nu is not None:
            reward_nu = -abs(nu - target_nu)
        
        # Combined reward (weighted sum)
        return 0.5 * reward_E + 0.5 * reward_nu
    
    def _clip_params(self, params):
        """Clip parameters to valid bounds."""
        clipped = {}
        for key in self.param_keys:
            lo, hi = self.bounds[key]
            clipped[key] = np.clip(params[key], lo, hi)
        return clipped
    
    def optimize(self, n_episodes=100, n_steps=20, learning_rate=0.1,
                 exploration=0.3, target_E=None, target_nu=None):
        """
        Run RL optimization with epsilon-greedy exploration.
        
        Returns history of parameters, rewards, and predicted properties.
        """
        history = {
            'episode': [], 'step': [],
            'angle': [], 'cell_height': [], 'cell_width': [], 'radius': [],
            'reward': [], 'E_pred': [], 'nu_pred': [],
            'best_reward': [],
        }
        
        best_reward = -np.inf
        best_params = None
        
        # Q-values for action selection (simplified)
        q_values = {a: 0.0 for a in self.action_space}
        
        for ep in range(n_episodes):
            # Random starting point
            params = {
                'angle': np.random.uniform(100, 170),
                'cell_height': np.random.uniform(6, 16),
                'cell_width': np.random.uniform(6, 16),
                'radius': np.random.uniform(0.1, 0.35),
            }
            
            ep_rewards = []
            
            for step in range(n_steps):
                # Epsilon-greedy action selection
                if np.random.random() < exploration:
                    action = self.action_space[np.random.randint(len(self.action_space))]
                else:
                    # Exploit: pick best known action
                    action = max(q_values, key=q_values.get)
                
                # Apply action (scaled to parameter ranges)
                new_params = {}
                scales = [2.0, 1.0, 1.0, 0.02]  # step sizes per param
                for i, key in enumerate(self.param_keys):
                    new_params[key] = params[key] + action[i] * scales[i]
                new_params = self._clip_params(new_params)
                
                # Compute reward
                reward = self._compute_reward(new_params, target_E, target_nu)
                E_pred, nu_pred = self._predict_properties(new_params)
                
                # Update Q-value
                q_values[action] += learning_rate * (reward - q_values[action])
                
                # Track best
                if reward > best_reward:
                    best_reward = reward
                    best_params = new_params.copy()
                
                # Record history
                for key in ['episode', 'step', 'reward', 'E_pred', 'nu_pred', 'best_reward']:
                    if key in ['episode', 'step']:
                        history[key].append(ep if key == 'episode' else step)
                    elif key == 'reward':
                        history[key].append(reward)
                    elif key == 'E_pred':
                        history[key].append(E_pred)
                    elif key == 'nu_pred':
                        history[key].append(nu_pred)
                    elif key == 'best_reward':
                        history[key].append(best_reward)
                
                for key in self.param_keys:
                    history[key].append(new_params[key])
                
                params = new_params
                ep_rewards.append(reward)
            
            # Decay exploration
            exploration = max(0.05, exploration * 0.98)
        
        return pd.DataFrame(history), best_params


# Initialize RL agent with trained ML models
rl_agent = MetamaterialRLAgent(
    ml_model_E=results_ml[("Random Forest", "log(E_x)")]['model'],
    ml_model_nu=results_ml[("Random Forest", "\u03bd_xy")]['model'],
    scaler=scaler,
    feature_names=all_features,
)

print(f"\u2705 RL agent initialized")
print(f"   Action space: {len(rl_agent.action_space)} discrete actions")
print(f"   Parameter bounds: {rl_agent.bounds}")
print(f"   Goal: maximize stiffness (E) + auxetic behavior (\u03bd < 0)")


In [ ]:
# ============================================================
# Run RL optimization
# ============================================================

print("Starting RL optimization...")
print("  Episodes: 100, Steps/episode: 20")
print("  Objective: high stiffness + negative Poisson's ratio")
print("-" * 60)

rl_history, best_rl_params = rl_agent.optimize(
    n_episodes=100,
    n_steps=20,
    learning_rate=0.1,
    exploration=0.3,
)

print(f"\n\ud83c\udfc6 Best parameters found by RL:")
for key, val in best_rl_params.items():
    if isinstance(val, float):
        print(f"   {key}: {val:.3f}")
    else:
        print(f"   {key}: {val}")

E_best, nu_best = rl_agent._predict_properties(best_rl_params)
print(f"\n   ML-predicted: E = {E_best:.2e} Pa, \u03bd = {nu_best:.3f}")


In [ ]:
# ============================================================
# Visualize RL optimization trajectory
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# 1. Reward over training
ax = axes[0, 0]
ax.plot(rl_history['reward'], alpha=0.3, color='gray', label='Step reward')
ax.plot(rl_history['best_reward'], 'r-', linewidth=2, label='Best reward')
ax.set_xlabel('Training step')
ax.set_ylabel('Reward')
ax.set_title('RL Training: Reward Convergence')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Predicted modulus over training
ax = axes[0, 1]
ax.plot(rl_history['E_pred'], alpha=0.3, color='blue')
ax.set_xlabel('Training step')
ax.set_ylabel('Predicted E (Pa)')
ax.set_title('ML-Predicted Modulus over Training')
ax.grid(True, alpha=0.3)

# 3. Predicted Poisson's ratio
ax = axes[0, 2]
ax.plot(rl_history['nu_pred'], alpha=0.3, color='green')
ax.axhline(y=0, color='red', linestyle='--', label='\u03bd=0')
ax.set_xlabel('Training step')
ax.set_ylabel("Predicted \u03bd_xy")
ax.set_title("ML-Predicted Poisson's Ratio over Training")
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Parameter trajectory: angle
ax = axes[1, 0]
ax.plot(rl_history['angle'], alpha=0.3, color='purple')
ax.axhline(y=best_rl_params['angle'], color='red', linestyle='--',
           linewidth=2, label=f"Best: {best_rl_params['angle']:.1f}\u00b0")
ax.set_xlabel('Training step')
ax.set_ylabel('Re-entrant angle \u03b8 (\u00b0)')
ax.set_title('Parameter Trajectory: Angle')
ax.legend()
ax.grid(True, alpha=0.3)

# 5. Property space: E vs nu (trajectory)
ax = axes[1, 1]
colors_step = np.arange(len(rl_history))
sc = ax.scatter(rl_history['nu_pred'], rl_history['E_pred'],
                c=colors_step, cmap='coolwarm', s=20, alpha=0.5)
plt.colorbar(sc, ax=ax, label='Training step')
ax.scatter([nu_best], [E_best], c='red', s=200, marker='*',
           edgecolors='black', linewidth=1.5, label='RL Best', zorder=5)
ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel("Predicted \u03bd_xy")
ax.set_ylabel('Predicted E (Pa)')
ax.set_title('Property Space Exploration')
ax.legend()
ax.grid(True, alpha=0.3)

# 6. Best structure
ax = axes[1, 2]
try:
    net_best = reentrant_honeycomb_2d(
        reentrant_angle=best_rl_params['angle'],
        grid_size=(4, 4),
        radius=best_rl_params['radius'],
    )
    for fiber in net_best.fibers:
        pts = fiber.centerline
        ax.plot(pts[:, 0], pts[:, 1], 'b-', alpha=0.7, linewidth=1.5)
    ax.set_title(f"RL-Optimized Structure\n"
                 f"\u03b8={best_rl_params['angle']:.1f}\u00b0, "
                 f"r={best_rl_params['radius']:.2f}")
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
except Exception as e:
    ax.text(0.5, 0.5, f'Error: {e}', ha='center', va='center')

plt.suptitle('Reinforcement Learning: Closed-Loop Metamaterial Design',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("fibernet_output/results/rl_optimization.png",
            dpi=200, bbox_inches='tight', facecolor='white')
print("  \ud83d\udcbe Saved: fibernet_output/results/rl_optimization.png")
plt.show()


In [ ]:
# ============================================================
# Targeted RL: find structure with specific target properties
# ============================================================

# Target: E = 1e7 Pa, nu = -0.3 (moderately auxetic, specific stiffness)
target_E = 1e7
target_nu = -0.3

print(f"Targeted RL optimization:")
print(f"  Target E  = {target_E:.2e} Pa")
print(f"  Target \u03bd  = {target_nu:.1f}")
print("-" * 60)

targeted_history, targeted_best = rl_agent.optimize(
    n_episodes=80,
    n_steps=15,
    learning_rate=0.15,
    exploration=0.25,
    target_E=target_E,
    target_nu=target_nu,
)

E_found, nu_found = rl_agent._predict_properties(targeted_best)

print(f"\n\ud83c\udfaf Best found:")
for key, val in targeted_best.items():
    if isinstance(val, float):
        print(f"   {key}: {val:.3f}")
print(f"   Predicted E = {E_found:.2e} Pa (target: {target_E:.2e})")
print(f"   Predicted \u03bd = {nu_found:.3f} (target: {target_nu:.1f})")
print(f"   Error E: {abs(E_found - target_E)/target_E:.2%}")
print(f"   Error \u03bd: {abs(nu_found - target_nu):.3f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(targeted_history['E_pred'], alpha=0.4, color='blue')
ax1.axhline(y=target_E, color='red', linestyle='--', linewidth=2, label='Target E')
ax1.set_xlabel('Step')
ax1.set_ylabel('Predicted E (Pa)')
ax1.set_title('Targeted RL: Modulus Convergence')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(targeted_history['nu_pred'], alpha=0.4, color='green')
ax2.axhline(y=target_nu, color='red', linestyle='--', linewidth=2, label='Target \u03bd')
ax2.set_xlabel('Step')
ax2.set_ylabel("Predicted \u03bd")
ax2.set_title("Targeted RL: Poisson's Ratio Convergence")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("fibernet_output/results/rl_targeted.png",
            dpi=200, bbox_inches='tight', facecolor='white')
print("  \ud83d\udcbe Saved: fibernet_output/results/rl_targeted.png")
plt.show()


## 9. FEM Validation & Verification / 有限元验证

To ensure our FEM results are trustworthy, we validate against analytical solutions.

### How FiberNet's Mechanical Simulation Works

FiberNet implements a **custom finite element solver** built from scratch using NumPy + SciPy.
It does **not** wrap or depend on any open-source Python FEM library (FEniCS, SfePy, PyFEM, etc.).

| Component | Implementation |
|-----------|---------------|
| **Element type** | 3D Euler-Bernoulli beam (12 DOF/element: 3 translations + 3 rotations per node) |
| **Stiffness matrix** | Analytical 12×12 local stiffness with coordinate transformation |
| **Sparse solver** | `scipy.sparse.linalg.spsolve` (SuperLU direct solver) |
| **Regularization** | Tikhonov regularization for near-singular systems |
| **Nonlinear** | Newton-Raphson iteration with incremental loading |
| **Constitutive models** | Linear elastic, bilinear plasticity, Neo-Hookean, Mooney-Rivlin, Arruda-Boyce |

### Validation Tests

1. **Cantilever beam** vs. Euler-Bernoulli analytical solution (δ = PL³/3EI)
2. **Patch test** (uniform strain reproduction)
3. **Gibson-Ashby scaling** (E* ∝ ρ³ for honeycomb)
4. **h-refinement convergence** (mesh sensitivity)

In [ ]:
# ============================================================
# Validation 1: Cantilever Beam vs Analytical Solution
# ============================================================

from fibernet.sim.validation import (
    validate_cantilever_beam,
    euler_beam_cantilever,
)

# Beam parameters
E_beam = 200e9   # Steel
L_beam = 1.0     # 1 meter
r_beam = 0.01    # 10 mm radius
P_beam = 100.0   # 100 N tip load
I_beam = np.pi * r_beam**4 / 4

# Analytical solution
analytical = euler_beam_cantilever(L=L_beam, E=E_beam, I=I_beam, P=P_beam)
print("Cantilever Beam Validation")
print("=" * 60)
print(f"  EI = {E_beam * I_beam:.4e} N\u00b7m\u00b2")
print(f"  Analytical \u03b4_tip = {analytical['delta_tip']:.6e} m")
print(f"  Analytical \u03b8_tip = {analytical['theta_tip']:.6e} rad")
print(f"  Analytical M_max  = {analytical['M_max']:.1f} N\u00b7m")
print()

# Convergence study: refine mesh and measure error
seg_counts = [2, 4, 8, 16, 32]
convergence_data = []

for n_seg in seg_counts:
    result = validate_cantilever_beam(
        L=L_beam, E=E_beam, r=r_beam, P=P_beam,
        segments=n_seg, tolerance=1.0
    )
    convergence_data.append({
        'segments': n_seg,
        'numerical': result.numerical_value,
        'analytical': result.analytical_value,
        'error': result.relative_error,
    })
    status = '\u2705' if result.passed else '\u274c'
    print(f"  {status} segments={n_seg:2d}: "
          f"\u03b4={result.numerical_value:.6e} m, "
          f"error={result.relative_error:.4%}")

# Plot convergence
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

errors = [d['error'] for d in convergence_data]
ax1.semilogy(seg_counts, errors, 'o-', linewidth=2, markersize=10, color='steelblue')
ax1.axhline(y=0.01, color='red', linestyle='--', linewidth=2, label='1% threshold')
ax1.set_xlabel('Segments per fiber', fontsize=13)
ax1.set_ylabel('Relative Error', fontsize=13)
ax1.set_title('FEM Convergence: Cantilever Beam', fontsize=14, fontweight='bold')
ax1.legend(fontsize=12)
ax1.grid(True, alpha=0.3)

# Beam deflection shape
x = np.linspace(0, L_beam, 100)
y_exact = (P_beam * x**2 / (6 * E_beam * I_beam)) * (3*L_beam - x)
ax2.plot(x, y_exact * 1000, 'b-', linewidth=2.5, label='Analytical')
ax2.set_xlabel('x (m)', fontsize=13)
ax2.set_ylabel('Deflection (mm)', fontsize=13)
ax2.set_title('Cantilever Beam Deflection', fontsize=14, fontweight='bold')
ax2.legend(fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("fibernet_output/results/validation_cantilever.png",
            dpi=200, bbox_inches='tight', facecolor='white')
print("\n  \ud83d\udcbe Saved: fibernet_output/results/validation_cantilever.png")
plt.show()


In [ ]:
# ============================================================
# Validation 2: Gibson-Ashby Honeycomb Scaling Law
# ============================================================

from fibernet.sim.validation import gibson_ashby_honeycomb

# Test the cubic scaling law: E*/Es ~ (t/l)^3 * constant
print("Gibson-Ashby Honeycomb Scaling Validation")
print("=" * 60)
print("Theory: E*/Es = C * (\u03c1*/\u03c1s)\u00b3  (bending-dominated, regular hexagonal)")
print()

rho_ratios = np.linspace(0.02, 0.25, 30)
E_solid = 1e9

# Analytical Gibson-Ashby predictions
E_ga_x = []
E_ga_y = []
for rho in rho_ratios:
    ga = gibson_ashby_honeycomb(relative_density=rho, E_solid=E_solid)
    E_ga_x.append(ga['E1'])
    E_ga_y.append(ga['E2'])

E_ga_x = np.array(E_ga_x)
E_ga_y = np.array(E_ga_y)

# Also show 3D foam scaling for comparison
E_foam = E_solid * rho_ratios**2  # stretching-dominated

# FEM verification at selected densities
from fibernet.gen.metamaterials import reentrant_honeycomb_2d

fem_points_rho = []
fem_points_E = []

test_densities = [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]
for rho_test in test_densities:
    try:
        # Adjust radius to approximate target density
        r_approx = rho_test * 0.4
        net_test = reentrant_honeycomb_2d(
            reentrant_angle=120,  # hexagonal (not auxetic)
            grid_size=(4, 4),
            radius=r_approx,
        )
        fem_test = FiberFEM(net_test, segments_per_fiber=3)
        E_fem = fem_test.effective_modulus(strain=0.001, axis=0)
        if E_fem > 0:
            fem_points_rho.append(rho_test)
            fem_points_E.append(E_fem)
            print(f"  \u03c1*/\u03c1s={rho_test:.2f}: E_FEM={E_fem:.2e} Pa")
    except Exception as e:
        print(f"  \u03c1*/\u03c1s={rho_test:.2f}: skipped ({e})")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
ax1.plot(rho_ratios, E_ga_x / E_solid, 'b-', linewidth=2.5,
         label='Gibson-Ashby E1 (analytical)')
ax1.plot(rho_ratios, E_ga_y / E_solid, 'g--', linewidth=2,
         label='Gibson-Ashby E2 (analytical)')
ax1.plot(rho_ratios, E_foam / E_solid, 'orange', linewidth=2, alpha=0.5,
         label='3D Foam (E \u221d \u03c1\u00b2)')
if fem_points_rho:
    ax1.scatter(fem_points_rho, [E/E_solid for E in fem_points_E],
                c='red', s=120, marker='*', edgecolors='black',
                linewidth=1, label='FiberNet FEM', zorder=5)
ax1.set_xlabel('Relative Density \u03c1*/\u03c1s', fontsize=13)
ax1.set_ylabel('E*/Es', fontsize=13)
ax1.set_title('Gibson-Ashby Scaling (Linear)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Log-log scale (shows power-law slopes clearly)
ax2.loglog(rho_ratios, E_ga_x / E_solid, 'b-', linewidth=2.5,
           label='GA E1 (slope\u22483)')
ax2.loglog(rho_ratios, E_ga_y / E_solid, 'g--', linewidth=2,
           label='GA E2')
ax2.loglog(rho_ratios, E_foam / E_solid, 'orange', linewidth=2, alpha=0.5,
           label='3D Foam (slope=2)')
if fem_points_rho:
    ax2.scatter(fem_points_rho, [E/E_solid for E in fem_points_E],
                c='red', s=120, marker='*', edgecolors='black',
                linewidth=1, label='FiberNet FEM', zorder=5)

# Reference lines for slopes
ref_rho = np.array([0.05, 0.2])
ax2.loglog(ref_rho, ref_rho**3 / ref_rho[0]**3 * E_ga_x[3]/E_solid,
           'k:', linewidth=1.5, alpha=0.5, label='slope=3')
ax2.loglog(ref_rho, ref_rho**2 / ref_rho[0]**2 * E_foam[3]/E_solid,
           'k--', linewidth=1.5, alpha=0.5, label='slope=2')

ax2.set_xlabel('Relative Density \u03c1*/\u03c1s', fontsize=13)
ax2.set_ylabel('E*/Es', fontsize=13)
ax2.set_title('Gibson-Ashby Scaling (Log-Log)', fontsize=14, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig("fibernet_output/results/validation_gibson_ashby.png",
            dpi=200, bbox_inches='tight', facecolor='white')
print("\n  \ud83d\udcbe Saved: fibernet_output/results/validation_gibson_ashby.png")
plt.show()


In [ ]:
# ============================================================
# Validation 3: FEM Verification of RL-Optimized Structure
# Compare ML prediction vs actual FEM result
# ============================================================

print("FEM Verification of RL-Optimized Design")
print("=" * 60)

# Generate the RL-optimal structure
try:
    net_rl = reentrant_honeycomb_2d(
        reentrant_angle=best_rl_params['angle'],
        grid_size=(4, 4),
        radius=best_rl_params['radius'],
    )
    
    # Run actual FEM
    fem_rl = FiberFEM(net_rl, segments_per_fiber=8)
    E_fem_rl = fem_rl.effective_modulus(strain=0.001, axis=0)
    nu_fem_rl = poisson_ratio(fem_rl, strain=0.001)
    
    # Compare with ML prediction
    E_ml, nu_ml = rl_agent._predict_properties(best_rl_params)
    
    print(f"\nRL-optimal parameters:")
    for k, v in best_rl_params.items():
        print(f"  {k}: {v:.3f}")
    
    print(f"\nProperty comparison:")
    print(f"  {'Property':<20s} {'ML Predicted':>15s} {'FEM Actual':>15s} {'Error':>10s}")
    print(f"  {'-'*60}")
    E_err = abs(E_fem_rl - E_ml) / E_ml * 100 if E_ml > 0 else float('inf')
    nu_err = abs(nu_fem_rl - nu_ml) if nu_ml != 0 else float('inf')
    print(f"  {'E_x (Pa)':<20s} {E_ml:>15.2e} {E_fem_rl:>15.2e} {E_err:>9.1f}%")
    print(f"  {'\u03bd_xy':<20s} {nu_ml:>15.3f} {nu_fem_rl:>15.3f} {nu_err:>10.3f}")
    
    # Visualize comparison
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Structure
    ax = axes[0]
    for fiber in net_rl.fibers:
        pts = fiber.centerline
        ax.plot(pts[:, 0], pts[:, 1], 'b-', alpha=0.7, linewidth=1.5)
    ax.set_title(f"RL-Optimized Structure\n\u03b8={best_rl_params['angle']:.1f}\u00b0, "
                 f"r={best_rl_params['radius']:.2f}")
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    
    # Deformation under strain
    ax = axes[1]
    result_def = fem_rl.apply_uniaxial_strain(strain=0.02, axis=0)
    for fiber in net_rl.fibers:
        pts = fiber.centerline
        ax.plot(pts[:, 0], pts[:, 1], 'b-', alpha=0.2, linewidth=0.5)
    # Plot deformed shape
    if result_def.displacements is not None:
        node_pos = fem_rl.node_positions
        u = result_def.displacements.reshape(-1, 6)
        scale = 5  # amplification
        for elem_nodes in fem_rl.element_connectivity:
            n1, n2 = elem_nodes
            p1 = node_pos[n1] + scale * u[n1, :3]
            p2 = node_pos[n2] + scale * u[n2, :3]
            ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 'r-', linewidth=1.5, alpha=0.7)
    ax.set_title(f"Deformation (5\u00d7 amplification)\n"
                 f"E_FEM={E_fem_rl:.2e} Pa")
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    
    # Stress-strain curve
    ax = axes[2]
    strains_ss, stresses_ss = stress_strain_curve(
        net_rl, max_strain=0.05, num_steps=10, axis=0, segments_per_fiber=5
    )
    ax.plot(strains_ss * 100, stresses_ss / 1e6, 'o-', linewidth=2, color='steelblue')
    ax.set_xlabel('Strain (%)')
    ax.set_ylabel('Stress (MPa)')
    ax.set_title('Stress-Strain of RL-Optimized Design')
    ax.grid(True, alpha=0.3)
    
    plt.suptitle('Validation: RL-Optimized Metamaterial (FEM Verification)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig("fibernet_output/results/validation_rl_fem.png",
                dpi=200, bbox_inches='tight', facecolor='white')
    print("\n  \ud83d\udcbe Saved: fibernet_output/results/validation_rl_fem.png")
    plt.show()
    
except Exception as e:
    print(f"Error during FEM validation: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
# ============================================================
# Run Complete Validation Suite
# ============================================================

from fibernet.sim.validation import (
    run_all_validations,
    print_validation_report,
)

print("Running complete FEM validation suite...")
print()

val_results = run_all_validations(E_solid=1e9, verbose=True)
report = print_validation_report(val_results)
print(report)

n_pass = sum(1 for r in val_results if r.passed)
print(f"\n\ud83d\udcca Overall: {n_pass}/{len(val_results)} tests passed")


## 10. Summary / 总结

### Complete Pipeline

1. ✅ **Installed** FiberNet
2. ✅ **Generated** 9+ metamaterial structures (auxetic, chiral, star, arrowhead, octet, diamond, hierarchical, woven, braided)
3. ✅ **Visualized** all structures with publication-quality matplotlib plots
4. ✅ **Saved** structures as PNG + JSON
5. ✅ **Parametric study**: Swept re-entrant angle, computed E*, ν*, ρ*
6. ✅ **FEM simulation**: Stress-strain curves, deformation visualization
7. ✅ **ML surrogate**: Trained RF/GBM/MLP models, screened 500 candidates
8. ✅ **Reinforcement Learning**: Closed-loop optimization using ML surrogate as reward
9. ✅ **Validation**: Cantilever beam, Gibson-Ashby scaling, FEM verification of RL design

### Closed-Loop Architecture

```
Structure Generation → FEM Simulation → Feature Extraction → ML Training
        ↑                                                            ↓
        └── RL Agent proposes designs ← ML surrogate as reward ←────┘
                                  ↓
                        FEM validates best design
```

### How the Mechanical Simulation Works

FiberNet uses a **custom FEM implementation** (not an open-source FEM library):

- **Element**: 3D Euler-Bernoulli beam (12 DOF/element)
- **Solver**: scipy.sparse + SuperLU direct solver
- **Validated against**: Euler-Bernoulli cantilever beam (δ = PL³/3EI), Gibson-Ashby cellular solid theory

### File Structure

```
fibernet_output/
├── structures/           # PNG + JSON for each structure
├── results/              # Analysis outputs
│   ├── parametric_study.png
│   ├── stress_strain_curve.png
│   ├── ml_results.png
│   ├── design_screening.png
│   ├── rl_optimization.png     # NEW
│   ├── rl_targeted.png          # NEW
│   ├── validation_cantilever.png # NEW
│   ├── validation_gibson_ashby.png # NEW
│   └── validation_rl_fem.png    # NEW
└── dataset/              # Raw data for ML
```

### Next Steps

- Use **stable-baselines3** (PPO/SAC) for more sophisticated RL
- Extend to **3D structures** with periodic boundary conditions
- Add **multi-physics** coupling (thermo-mechanical, piezoresistive)
- Combine with **experimental data** for model calibration

### Further Resources

- [FiberNet Documentation](https://fibernet.readthedocs.io/)
- [GitHub Repository](https://github.com/GellmanSparrowS/fibernet)
- [FEM Implementation Details](https://github.com/GellmanSparrowS/fibernet/blob/main/docs/fem_implementation.md)
